In [5]:
import torch
import torch.nn as nn

# ──────────────────────────────────────────────
# 1. Numerical gradient (finite differences)
# ──────────────────────────────────────────────

def numerical_gradient(model: nn.Module, loss_fn, x: torch.Tensor,
                       y: torch.Tensor, eps: float = 1e-5) -> dict:
    """
    Compute gradients numerically using the two-sided finite difference:

        dL/dθ ≈ [ L(θ + ε) - L(θ - ε) ] / 2ε

    This is slow but reliable — the gold standard for checking.
    """
    numerical_grads = {}

    for name, param in model.named_parameters():
        grad = torch.zeros_like(param)
        param_flat = param.data.view(-1)

        for idx in range(param_flat.numel()):
            # Save original value
            original = param_flat[idx].item()

            # L(θ + ε)
            param_flat[idx] = original + eps
            loss_plus = loss_fn(model(x), y).item()

            # L(θ - ε)
            param_flat[idx] = original - eps
            loss_minus = loss_fn(model(x), y).item()

            # Restore original value
            param_flat[idx] = original

            # Central difference
            grad.view(-1)[idx] = (loss_plus - loss_minus) / (2 * eps)

        numerical_grads[name] = grad

    return numerical_grads


# ──────────────────────────────────────────────
# 2. Analytical gradient (autograd)
# ──────────────────────────────────────────────

def analytical_gradient(model: nn.Module, loss_fn, x: torch.Tensor,
                        y: torch.Tensor) -> dict:
    """
    Compute gradients analytically using PyTorch autograd.
    This is what .backward() does — fast and exact.
    """
    model.zero_grad()
    loss = loss_fn(model(x), y)
    loss.backward()

    return {name: param.grad.clone() for name, param in model.named_parameters()}


# ──────────────────────────────────────────────
# 3. Gradient checker
# ──────────────────────────────────────────────

def check_gradients(model: nn.Module, loss_fn, x: torch.Tensor,
                    y: torch.Tensor, eps: float = 1e-5,
                    tol: float = 1e-5) -> dict:
    """
    Compare numerical and analytical gradients.

    Uses relative error:
        error = |num - ana| / max(|num|, |ana|, 1e-8)

    Rules of thumb:
        error < 1e-5  →  GREAT
        error < 1e-3  →  OK (might be fine for some activations)
        error > 1e-3  →  BUG likely
    """
    # ── Use float64 for better numerical precision ──
    original_dtype = next(model.parameters()).dtype
    model.double()
    x_d = x.double()
    y_d = y.double()

    # Analytical FIRST (needs autograd graph)
    ana_grads = analytical_gradient(model, loss_fn, x_d, y_d)

    # Numerical SECOND (uses torch.no_grad, no graph interference)
    num_grads = numerical_gradient(model, loss_fn, x_d, y_d, eps)

    results = {}

    for name in num_grads:
        num = num_grads[name]
        ana = ana_grads[name]

        # Relative error (safe against division by zero)
        diff  = torch.abs(num - ana)
        scale = torch.maximum(torch.abs(num), torch.abs(ana)).clamp(min=1e-8)
        rel_error = (diff / scale).max().item()

        passed = rel_error < tol

        results[name] = {
            "numerical":  num,
            "analytical": ana,
            "max_rel_error": rel_error,
            "passed": passed,
        }

    # Restore original dtype
    if original_dtype == torch.float32:
        model.float()

    return results


# ──────────────────────────────────────────────
# 4. Pretty printer
# ──────────────────────────────────────────────

def print_results(results: dict):
    """Display gradient check results in a readable table."""
    all_pass = True
    for name, r in results.items():
        status = "PASS ✓" if r["passed"] else "FAIL ✗"
        if not r["passed"]:
            all_pass = False
        print(f"  {name:<25}  error: {r['max_rel_error']:.2e}  [{status}]")

        # Show actual values for small params
        if r["numerical"].numel() <= 6:
            num_list = [f"{v:.8f}" for v in r["numerical"].flatten().tolist()]
            ana_list = [f"{v:.8f}" for v in r["analytical"].flatten().tolist()]
            print(f"    numerical:  [{', '.join(num_list)}]")
            print(f"    analytical: [{', '.join(ana_list)}]")

    print(f"\n  Overall: {'ALL PASSED ✓' if all_pass else 'SOME FAILED ✗'}\n")


# ──────────────────────────────────────────────
# 5. Example usage
# ──────────────────────────────────────────────

if __name__ == "__main__":
    print("=" * 55)
    print("  Gradient Checking: Numerical vs Analytical")
    print("=" * 55)

    torch.manual_seed(42)

    # ── Test 1: Single neuron (simple) ──
    print("\n── Test 1: Single sigmoid neuron ──\n")

    model1 = nn.Sequential(nn.Linear(3, 1), nn.Sigmoid())
    loss_fn = nn.MSELoss()

    x1 = torch.randn(1, 3)
    y1 = torch.tensor([[1.0]])

    results1 = check_gradients(model1, loss_fn, x1, y1)
    print_results(results1)

    # ── Test 2: Multi-layer network ──
    print("── Test 2: Multi-layer network (3 → 8 → 4 → 1) ──\n")

    model2 = nn.Sequential(
        nn.Linear(3, 8), nn.ReLU(),
        nn.Linear(8, 4), nn.ReLU(),
        nn.Linear(4, 1), nn.Sigmoid(),
    )
    loss_fn2 = nn.BCELoss()

    x2 = torch.randn(4, 3)     # batch of 4
    y2 = torch.tensor([[1.0], [0.0], [1.0], [0.0]])

    results2 = check_gradients(model2, loss_fn2, x2, y2)
    print_results(results2)

    # ── Test 3: Intentionally WRONG gradient (should FAIL) ──
    print("── Test 3: Buggy custom layer (should FAIL) ──\n")

    class BuggyLinear(nn.Module):
        """Linear layer with a WRONG backward (doubles the gradient)."""
        def __init__(self, in_f, out_f):
            super().__init__()
            self.weight = nn.Parameter(torch.randn(out_f, in_f) * 0.1)
            self.bias   = nn.Parameter(torch.zeros(out_f))

        def forward(self, x):
            out = x @ self.weight.t() + self.bias
            # Register a bad backward hook that corrupts gradients
            out.register_hook(lambda g: g * 2.0)   # ← BUG: doubles gradients
            return out

    model3 = nn.Sequential(BuggyLinear(3, 1), nn.Sigmoid())

    results3 = check_gradients(model3, nn.MSELoss(), x1, y1)
    print_results(results3)

    # ── Test 4: Different epsilon values ──
    print("── Test 4: Effect of epsilon on accuracy ──\n")

    model4 = nn.Sequential(nn.Linear(3, 1), nn.Sigmoid())

    for eps in [1e-2, 1e-3, 1e-5, 1e-7, 1e-10]:
        results = check_gradients(model4, nn.MSELoss(), x1, y1, eps=eps)
        worst = max(r["max_rel_error"] for r in results.values())
        status = "PASS" if worst < 1e-5 else "meh " if worst < 1e-3 else "FAIL"
        print(f"  eps={eps:.0e}  →  worst error: {worst:.2e}  [{status}]")

    print()
    print("  → eps=1e-5 is the sweet spot (default)")
    print("  → Too large = bad approximation, too small = float rounding errors")

    # ── Summary ──
    print(f"""
── Gradient checking cheat sheet ──

  error < 1e-7  →  perfect
  error < 1e-5  →  great (normal target)
  error < 1e-3  →  suspicious (check kinks like ReLU at 0)
  error > 1e-3  →  bug in your backward pass

  Use it to debug custom layers, then REMOVE it for training
  (numerical gradients are ~1000× slower than autograd).
""")

  Gradient Checking: Numerical vs Analytical

── Test 1: Single sigmoid neuron ──

  0.weight                   error: 8.74e-11  [PASS ✓]
    numerical:  [-0.02797716, -0.02748443, 0.13398454]
    analytical: [-0.02797716, -0.02748443, 0.13398454]
  0.bias                     error: 1.96e-11  [PASS ✓]
    numerical:  [-0.11932474]
    analytical: [-0.11932474]

  Overall: ALL PASSED ✓

── Test 2: Multi-layer network (3 → 8 → 4 → 1) ──

  0.weight                   error: 3.59e-07  [PASS ✓]
  0.bias                     error: 2.12e-09  [PASS ✓]
  2.weight                   error: 3.22e-08  [PASS ✓]
  2.bias                     error: 7.95e-10  [PASS ✓]
    numerical:  [0.00000000, -0.01327842, 0.00556790, 0.02418320]
    analytical: [0.00000000, -0.01327842, 0.00556790, 0.02418320]
  4.weight                   error: 1.92e-09  [PASS ✓]
    numerical:  [0.00000000, 0.00286521, -0.00756242, -0.06382825]
    analytical: [0.00000000, 0.00286521, -0.00756242, -0.06382825]
  4.bias           